# Data Pipeline

## Imports

In [3]:
import os
import json
from pathlib import Path
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import AzureOpenAI
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv

## Load Environment Variables

In [5]:
load_dotenv("../.env")

True

## Helper Functions

In [6]:
def get_reference(text, deployment = "gpt-5.4", api_version = "2024-12-01-preview"):

    sys_prompt = """You are an AI assistant that extracts the bibliographic reference of a scientific paper itself by analyzing its first page. 
Your task is to identify the paper’s own citation details and return them in JSON format with the following fields:

{
  "reference": {
    "title": "string or N/A",
    "first_author": "string or N/A",
    "year": "string or N/A",
    "publication": "string or N/A"
  }
}

Rules:
- Always output valid JSON only, with no extra commentary.
- "title" is the title of the paper.
- "first_author" is the first listed author of the paper.
- "year" is the publication year of the paper.
- "publication" is the journal, conference, or book where the paper was published.
- If any field cannot be found on the first page, set its value to "N/A".
- Do not attempt to extract references cited by the paper; only extract the reference of the paper itself.
"""

    client = AzureOpenAI(
        api_version=api_version,
        azure_endpoint=os.environ["AZURE_ENDPOINT"],
        api_key=os.environ["AZURE_API_KEY"],
    )

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": text,
            }
        ],
        max_completion_tokens=2048,
        model=deployment
    )

    return json.loads(response.choices[0].message.content)

def format_reference(reference):
    _r = reference["reference"]
    return f"{_r['title']}, {_r['first_author']}, {_r['year']}"

def create_chunks(documents, references):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2500,    # Max characters per chunk, in average an english word is 5 characters
        chunk_overlap=0,  # Characters to repeat between chunks to keep context
        length_function=len,
        is_separator_regex=False,
    )

    chunks = []

    for doc, ref in zip(documents, references):
        ref_string = format_reference(ref)
        for page in doc:
            page_ref_string = ref_string + f", page {page['metadata']['page_number']}."
            _chunks = text_splitter.split_text(page["text"])
            _chunks = [f"Source: {page_ref_string}\n---\n{c}" for c in _chunks]
            chunks.extend(_chunks)
    
    return chunks

In [7]:
def prepare_pdf_chunks(file_dir):
    
    documents = []
    references = []
    
    for file in Path(file_dir).glob("*.pdf"):
        print(f"Parsing file: {file}")
        pages = pymupdf4llm.to_markdown(
            file,
            force_ocr=True,
            ocr_dpi=300,
            dpi=300,
            page_chunks=True,
            table_strategy="lines_strict",
            ocr_language="eng",
            header=False,
            footer=False,
        )
        documents.append(pages)

    for document in documents:
        reference = get_reference(document[0]["text"])
        references.append(reference)
    
    chunks = create_chunks(documents, references)

    return chunks


In [8]:
file_dir = "../../data"

chunks = prepare_pdf_chunks(file_dir)

Parsing file: ..\..\data\attention_paper.pdf
Parsing file: ..\..\data\gemini_paper.pdf
Parsing file: ..\..\data\gpt4.pdf
Parsing file: ..\..\data\instructgpt.pdf
Parsing file: ..\..\data\mistral_paper.pdf


In [12]:
model = SentenceTransformer("BAAI/bge-m3")

def get_dense_emb(text):

    embeddings = model.encode([text])

    return embeddings[0].tolist()


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 23121.66it/s]


In [13]:
points = []

for i, text in enumerate(chunks):
    # 1. Generate Embeddings
    dense_vector = get_dense_emb(text)
    
    # 2. Map to Qdrant Point structure
    points.append(
        models.PointStruct(
            id=i,
            vector={
                "dense": dense_vector,
            },
            payload={
                "text": text,
                # Add any other metadata here
            }
        )
    )


In [14]:
# Initialize client (assuming collection is already created as shown previously)
client = QdrantClient(
    url=os.environ["QDRANT_ENDPOINT"],
    api_key=os.environ["QDRANT_API_KEY"]
)

# Assuming 'points' is your list of PointStructs
batch_size = 50 

for i in range(0, len(points), batch_size):
    batch = points[i : i + batch_size]
    client.upsert(
        collection_name="bge_rag_database",
        points=batch
    )
    print(f"Uploaded batch {i // batch_size + 1}")


Uploaded batch 1
Uploaded batch 2
Uploaded batch 3
Uploaded batch 4


In [21]:
query_text = "What can you tell me about Attention?"

# 1. Generate query embeddings using your functions
dense_query = get_dense_emb(query_text)

# 2. Execute Hybrid Search (RRF)
results = client.query_points(
    collection_name="bge_rag_database",
    using="dense",
    query=dense_query,
    limit=50
)

# Display results
for point in results.points:
    print(f"ID: {point.id}, Score: {point.score}, Text: {point.payload['text'][:100]}...")


ID: 24, Score: 0.6063998, Text: Source: Attention Is All You Need, Ashish Vaswani, 2017, page 14.
---
**==> picture [383 x 458] inte...
ID: 25, Score: 0.60457337, Text: Source: Attention Is All You Need, Ashish Vaswani, 2017, page 15.
---
**==> picture [380 x 435] inte...
ID: 4, Score: 0.5968635, Text: Source: Attention Is All You Need, Ashish Vaswani, 2017, page 3.
---
**==> picture [220 x 323] inten...
ID: 7, Score: 0.587574, Text: Source: Attention Is All You Need, Ashish Vaswani, 2017, page 5.
---
output values. These are concat...
ID: 0, Score: 0.57400215, Text: Source: Attention Is All You Need, Ashish Vaswani, 2017, page 1.
---
# **Attention Is All You Need**...
ID: 5, Score: 0.571265, Text: Source: Attention Is All You Need, Ashish Vaswani, 2017, page 4.
---
Scaled Dot-Product Attention 

...
ID: 17, Score: 0.57090974, Text: Source: Attention Is All You Need, Ashish Vaswani, 2017, page 10.
---
Table 4: The Transformer gener...
ID: 23, Score: 0.5673318, Text: Source: Attention I

In [22]:
from rerankers import Reranker

# 1. Initialize the model
reranker = Reranker("BAAI/bge-reranker-v2-m3", model_type="cross-encoder")

passages = [point.payload["text"] for point in results.points]

# 2. Rank the Passages
results = reranker.rank(query=query_text, docs=passages)

# 3. Print the sorted results
for result in results:
    print(f"Rank {result.rank} (Score: {result.score:.4f}): {result.text}")


Loading TransformerRanker model BAAI/bge-reranker-v2-m3 (this message can be suppressed by setting verbose=0)
No device set
Using device cpu
No dtype set
Using dtype torch.float32


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5858.74it/s]


Loaded model BAAI/bge-reranker-v2-m3
Using device cpu.
Using dtype torch.float32.
Rank 1 (Score: -3.2441): Source: Attention Is All You Need, Ashish Vaswani, 2017, page 5.
---
output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. 

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. 

MultiHead( _Q, K, V_ ) = Concat(head1 _, ...,_ headh) _W[O]_ 

**==> picture [202 x 14] intentionally omitted <==**

Where the projections are parameter matrices _Wi[Q] ∈_ R _[d]_[model] _[×][d][k]_ , _Wi[K] ∈_ R _[d]_[model] _[×][d][k]_ , _Wi[V] ∈_ R _[d]_[model] _[×][d][v]_ and _W[O] ∈_ R _[hd][v][×][d]_[model] . 

In this work we employ _h_ = 8 parallel attention layers, or heads. For each of these we use _dk_ = _dv_ = _d_ model _/h_ = 64. Due to the reduced dimension of each head, the total computatio